*This notebook is from https://github.com/neubig/anlp-code* by Graham Neubig

# Build a Rule-based Sentiment Classifier

This is a notebook for [CMU CS11-711 Advanced NLP](https://cmu-l3.github.io/anlp-fall2025/), in which you can attempt to build a rule-based sentiment classifier. It will take in a text `X` and return a `label` of "1" if the sentiment of the text is positive, "-1" if the sentiment of the text is negative, and "0" if the sentiment of the text is neutral. You can test the accuracy of your classifier on the [Stanford Sentiment Treebank](http://nlp.stanford.edu/sentiment/index.html) by running the notebook all the way to end.

The only thing that you should change in this notebook is the following cell which contains two important elements. The first is `extract_features(X)`, which will extract a dictionary of (named) feature values from the text. You should create this by hand, and a simple example is shown for you. The second is `feature_weights`, a dictionary which will assign a weight to each extracted feature.

The final way the classifier decides whether to assign a positive, negative, or neutral label is by calculating the dot product `feature_weights * extract_features(X)`, and if the value is greater than zero, return 1, less than zero return -1, and if exactly zero return 0.

Let's have some fun trying to design a classifier 😁

# 构建基于规则的情感分类器

这是一个用于 [CMU CS11-711 Advanced NLP](https://cmu-l3.github.io/anlp-fall2025/) 的 notebook，你可以尝试构建一个基于规则的情感分类器。它会接收文本 `X`，并返回一个标签 `label`：如果文本情感为积极，则返回 "1"；如果情感为消极，则返回 "-1"；如果情感为中性，则返回 "0"。你可以在运行整个 notebook 到最后时，测试这个分类器在 [Stanford Sentiment Treebank](http://nlp.stanford.edu/sentiment/index.html) 上的准确率。

在这个 notebook 中，你唯一需要修改的地方是下面这个单元格，其中包含两个重要元素。第一个是 `extract_features(X)`，它会从文本中提取一个（命名的）特征值字典。你需要手工创建它，并且已经给出了一个简单示例。第二个是 `feature_weights`，它是一个字典，用于为每个提取出的特征分配权重。

分类器决定是否分配积极、消极或中性标签的最终方式，是通过计算点积 `feature_weights * extract_features(X)`：如果值大于 0，则返回 1；如果小于 0，则返回 -1；如果恰好等于 0，则返回 0。

让我们一起设计一个分类器，并尽情享受其中的乐趣 😁

In [12]:
def extract_features(x: str) -> dict[str, float]:
    features = {}
    x_split = x.split(' ')
    
    # Count the number of "good words" and "bad words" in the text
    good_words = ['love', 'good', 'nice', 'great', 'enjoy', 'enjoyed']
    bad_words = ['hate', 'bad', 'terrible', 'disappointing', 'sad', 'lost', 'angry']
    for x_word in x_split:
        if x_word in good_words:
            features['good_word_count'] = features.get('good_word_count', 0) + 1
        if x_word in bad_words:
            features['bad_word_count'] = features.get('bad_word_count', 0) + 1
    
    # The "bias" value is always one, to allow us to assign a "default" score to the text
    features['bias'] = 1
    
    return features

feature_weights = {'good_word_count': 1.0, 'bad_word_count': -1.0, 'bias': 0.5}

In [ ]:
# Extract features from a string (I try to recall)
# def ectract_features(x: str) -> dict[str, float]: 输入语句(string)，输出字典{特征；数值}
#     features = {} 特征字典
#     x_split = x.split(' ') 分离语句

#     两个元组包含好坏词 自定义
#     good_words = []
#     bad_words = []

#     for x_word in x_xsplit: 迭代查找
#         if x_word in good_words:
#             feartures['good'] = features.get('good', 0) + 1 go0d类特征累和
#         if x_word in bad_words:
#             features['bad'] = features.get('bad', 0) + 1 bad类
    
#     features['bias'] = 1 偏置(?为何要有)

#     return features

## Data Reading

Read in the data from the training and dev (or finally test) sets

In [13]:
def read_xy_data(filename: str) -> tuple[list[str], list[int]]:
    x_data = []
    y_data = []
    with open(filename, 'r') as f:
        for line in f:
            label, text = line.strip().split(' ||| ')
            x_data.append(text)
            y_data.append(int(label))
    return x_data, y_data

In [ ]:
x_train, y_train = read_xy_data('./data/train.txt')
x_test, y_test = read_xy_data('./data/dev.txt')

In [15]:
print(x_train[0])
print(y_train[0])

The Rock is destined to be the 21st Century 's new `` Conan '' and that he 's going to make a splash even greater than Arnold Schwarzenegger , Jean-Claud Van Damme or Steven Segal .
1


## Run the Classifier and Calculate Accuracy

Run the classifier over the training and dev (test) sets and calculate accuracy

In [ ]:
def run_classifier(x: str) -> int:#判断这句话的情感1，-1，0
    score = 0
    for feat_name, feat_value in extract_features(x).items():
        score = score + feat_value * feature_weights.get(feat_name, 0)
    if score > 0:
        return 1
    elif score < 0:
        return -1
    else:
        return 0

In [ ]:
def calculate_accuracy(x_data: list[str], y_data: list[int]) -> float:
    total_number = 0
    correct_number = 0
    for x, y in zip(x_data, y_data):
        y_pred = run_classifier(x)  # 得出句子情感
        total_number += 1
        if y == y_pred:             # 情感与label match
            correct_number += 1
    return correct_number / float(total_number)

In [ ]:
label_count = {}    # 记录测试集中情感数
for y in y_test:
    if y not in label_count:
        label_count[y] = 0
    label_count[y] += 1
print(label_count)

{1: 444, 0: 229, -1: 428}


In [19]:
train_accuracy = calculate_accuracy(x_train, y_train)
test_accuracy = calculate_accuracy(x_test, y_test)
print(f'Train accuracy: {train_accuracy}')
print(f'Dev/test accuracy: {test_accuracy}')

Train accuracy: 0.4345739700374532
Dev/test accuracy: 0.4214350590372389


## Error Analysis

An important part of improving any system is figuring out where it goes wrong. The following two functions allow you to randomly observe some mistaken examples, which may help you improve the classifier. Feel free to write more sophisticated methods for error analysis as well.

In [20]:
import random
def find_errors(x_data, y_data):
    error_ids = []
    y_preds = []
    for i, (x, y) in enumerate(zip(x_data, y_data)):
        y_preds.append(run_classifier(x))
        if y != y_preds[-1]:
            error_ids.append(i)
    for _ in range(5):
        my_id = random.choice(error_ids)
        x, y, y_pred = x_data[my_id], y_data[my_id], y_preds[my_id]
        print(f'{x}\ntrue label: {y}\npredicted label: {y_pred}\n')

In [21]:
find_errors(x_train, y_train)

The thing looks like a made-for-home-video quickie .
true label: 0
predicted label: 1

Less front-loaded and more shapely than the two-hour version released here in 1990 .
true label: 0
predicted label: 1

A smug and convoluted action-comedy that does n't allow an earnest moment to pass without reminding audiences that it 's only a movie .
true label: -1
predicted label: 1

While it 's all quite tasteful to look at , the attention process tends to do a little fleeing of its own .
true label: -1
predicted label: 1

entertaining enough , but nothing new
true label: -1
predicted label: 1

